# PTBR-Reranker — Phase 2: hard negative mining (Kaggle, GPU free tier)

Valida o pipeline de Phase 2 com GPU T4 free tier:

1. Clona o repositório `tardellirs/ptbr-reranker`.
2. Instala dependências.
3. Baixa mMARCO-PT em modo `--small` (10k passagens, 1k train queries, 100 dev queries, qrels e triples oficiais).
4. Roda `data/mine_hard_negatives.py` em **GPU T4** usando o bi-encoder `PORTULAN/serafim-100m-portuguese-pt-sentence-encoder-ir`.
5. Roda `data/build_triples.py` para gerar o parquet final de treino, a partir de (a) triples oficiais MS MARCO (baseline) e (b) negativos minerados (upgrade).
6. Sanity checks no schema e conteúdo dos triples.

**Settings deste Kernel (Kaggle UI > Settings):**
- Accelerator: **GPU T4 x2** (free tier, 30h/semana). Single T4 também é suficiente.
- Internet: **On**.
- Persistence: **Variables and files**.
- Language: Python 3.

Execution time esperado: 8–15 min (encoding de 10k passagens + 100 queries em T4).

## 1. Clonar repositório e instalar dependências

In [ ]:
!rm -rf /kaggle/working/ptbr-reranker
!git clone --depth 1 https://github.com/tardellirs/ptbr-reranker.git /kaggle/working/ptbr-reranker
%cd /kaggle/working/ptbr-reranker
!git log --oneline -1

In [ ]:
!pip install -q -e ".[dev]" 2>&1 | tail -5

In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Download em modo `--small` (mMARCO-PT + qrels + triples oficiais)

In [ ]:
!python data/download_mmarco.py --small 2>&1 | tail -15

In [ ]:
!ls -la data/raw/mmarco/

## 3. Sanity check: cobertura de qrels nas 100 dev queries

In [ ]:
import pandas as pd

dev_queries = pd.read_parquet("data/raw/mmarco/queries_dev_portuguese.parquet")
print(f"Dev queries: {len(dev_queries)}")

qrels = pd.read_csv(
    "data/raw/mmarco/qrels.dev.small.tsv",
    sep="\t",
    names=["qid", "x", "pid", "rel"],
)
print(f"Qrels lines: {len(qrels)}")
print(f"Qrels unique qids: {qrels['qid'].nunique()}")

our_qids = set(dev_queries["id"].tolist())
covered = qrels[qrels["qid"].isin(our_qids)]
print(f"Dev queries (out of 100) with qrels: {covered['qid'].nunique()}")
print(f"Total (qid, pid) positives covering them: {len(covered)}")

## 4. Mine hard negatives (GPU step)

Usa Serafim-IR para encodar as 10k passagens + 100 queries, constrói índice FAISS HNSW, busca top-50 por query, filtra positivos via qrels, samplea 5 hard negatives da faixa rank [5, 50). Os parâmetros são reduzidos vs. produção (top-200, num-neg 7, rank 10-100) porque o corpus aqui tem só 10k passagens — não há rank 100 com passagens hard.

Esperado: ~30s para encoding em T4 + <1min para indexar/buscar.

In [ ]:
!python data/mine_hard_negatives.py \
    --queries data/raw/mmarco/queries_dev_portuguese.parquet \
    --qrels data/raw/mmarco/qrels.dev.small.tsv \
    --collection data/raw/mmarco/collection_portuguese.parquet \
    --output data/processed/hard_negatives_small.parquet \
    --top-k 50 \
    --num-negatives 5 \
    --rank-min 5 \
    --rank-max 50 \
    --device auto \
    2>&1 | tail -30

## 5. Sanity check qualitativa do mining

Para uma query, mostrar a query, o positivo (do qrels) e os 5 negativos minerados. Espera-se que negativos sejam *plausíveis mas não relevantes* — superficialmente similares mas não respondem à query.

In [ ]:
queries_text = (
    pd.read_parquet("data/raw/mmarco/queries_dev_portuguese.parquet")
    .set_index("id")["text"]
    .to_dict()
)
collection_text = (
    pd.read_parquet("data/raw/mmarco/collection_portuguese.parquet")
    .set_index("id")["text"]
    .to_dict()
)
mined_df = pd.read_parquet("data/processed/hard_negatives_small.parquet")

sample = mined_df.iloc[0]
print(f"Query #{sample['query_id']}: {queries_text.get(sample['query_id'], '<missing>')}")
print()
print(f"POSITIVE (#{sample['positive_id']}):")
print(f"  {collection_text.get(sample['positive_id'], '<missing>')[:200]}")
print()
for i, neg_id in enumerate(sample["negative_ids"], 1):
    text = collection_text.get(int(neg_id), "<missing>")
    print(f"NEGATIVE {i} (#{neg_id}):")
    print(f"  {text[:200]}")
    print()

## 6. Inspecionar o dataset oficial MS MARCO triples (sanity)

No modo `--small`, baixamos apenas 1k train queries e 10k passages, mas
`triples.train.ids.small.tsv` referencia IDs de todo o universo MS MARCO
(~808k train queries, 8.8M passagens). Não tentamos rodar `build_triples`
com `--official-triples` aqui — o filtro de IDs não bate. Em produção
(modo `--full`), `build_triples` consome esse arquivo direto.

Sanity: confirmar que o TSV existe, tem 100k linhas, e mostrar uma amostra.

In [ ]:
import subprocess

import pandas as pd

triples_tsv = pd.read_csv(
    "data/raw/mmarco/triples.train.ids.small.tsv",
    sep="\t",
    names=["qid", "pos_id", "neg_id"],
    nrows=5,
)
print("Sample of 5 rows from triples.train.ids.small.tsv:")
print(triples_tsv.to_string())

total = (
    subprocess.check_output(["wc", "-l", "data/raw/mmarco/triples.train.ids.small.tsv"])
    .decode()
    .split()[0]
)
print(f"\nTotal lines downloaded (small mode capped at 100000): {total}")

## 7. Build triples — hardneg recipe (mined negatives)

Agora consome os negativos minerados via Serafim-IR. Esse é o pipeline *production* (config `train_hardneg.yaml`).

In [ ]:
!python data/build_triples.py \
    --queries data/raw/mmarco/queries_dev_portuguese.parquet \
    --collection data/raw/mmarco/collection_portuguese.parquet \
    --mined-triples data/processed/hard_negatives_small.parquet \
    --output data/processed/triples_hardneg.parquet \
    2>&1 | tail -10

In [ ]:
hardneg = pd.read_parquet("data/processed/triples_hardneg.parquet")
print(f"Hardneg triples: {len(hardneg)}")
print()
row = hardneg.iloc[0]
print(f"Query: {row['query_text']}")
print(f"Positive: {row['positive_text'][:200]}")
print(f"Negative: {row['negative_text'][:200]}")

## 8. Checklist final

Se todas as células acima passaram:

- [x] Download mMARCO-PT + qrels + triples oficiais.
- [x] Sanity de cobertura qrels.
- [x] Mineração hard negatives com Serafim-IR em T4 GPU.
- [x] Build triples a partir do dataset oficial (baseline).
- [x] Build triples a partir dos negativos minerados (hardneg).
- [x] Schema final: `(query_id, query_text, positive_text, negative_text)`.

Phase 2 validada. Próxima fase: `src/train.py` (Phase 3 — treino real do cross-encoder com W&B logging).

Registrar resultado no `docs/lab_notebook.md`.